In [1]:
import pandas as pd 
import requests 
import numpy as np 
from bs4 import BeautifulSoup
import json, os, time, pdb 
import sys 
import warnings, logging 
import itertools 
from tqdm import tqdm
from argparse import ArgumentParser
from util_funcs import * 
from selenium import webdriver 
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.common.by import By 
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from scraper import scrape_model

def close_out_driver(wd):
    wd.close()
    wd.quit()


headers = {'User-Agent': 
           'Mozilla/5.0 (X11; Linux x86_64)'+\
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

In [2]:
league = 180659 # Six Nations league code
season = 2025

match_scrape = scrape_model(
    league_set=[league], 
    season_set=[season], 
    update_type='single team',
    date_set=None
    # driver_type='testing'
)

test_data_pull = match_scrape.gather_season_teams(
    league,
    season=season
)

print("team data for season pulled")
team_data_join_back = test_data_pull[
    ['game_id', 'date', 'competition', 'season', 'stadium']
]

6
/rugby/team/_/id/9/france
/rugby/team/_/id/1/england
/rugby/team/_/id/3/ireland
/rugby/team/_/id/2/scotland
/rugby/team/_/id/20/italy
/rugby/team/_/id/4/wales
team data for season pulled


In [3]:
game_dfs = []
for game in tqdm(range(len(test_data_pull))):
    try:
        if type(test_data_pull['game_id'].iloc[game]) == type('tester'):
            game_dfs.append(match_scrape.get_match_stats(
                game_id=test_data_pull['game_id'].iloc[game],
                league_id=test_data_pull['league_id'].iloc[game],
            ))
    except Exception as e: 
        print(e)
        # pdb.set_trace()

100%|██████████| 15/15 [00:04<00:00,  3.04it/s]


In [4]:
all_teams_df = pd.concat(game_dfs, axis=0)
all_teams_df = all_teams_df.merge(team_data_join_back, how='left', on='game_id')
all_teams_df = match_scrape.clean_match_stats(all_teams_df)


In [5]:
all_teams_df.iloc[:5]

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_rucks_won_percent,away_mauls_won_num,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent
0,600264,180659,France,35,Scotland,16,4,1,3,1,...,0.96,5,5,1.00,5,7,0.71,15,16,0.93
1,600259,180659,Ireland,27,France,42,3,5,3,4,...,0.93,8,9,0.88,3,3,1.00,12,12,1.00
2,600258,180659,Italy,24,France,73,3,11,3,9,...,0.99,7,8,0.87,5,5,1.00,15,15,1.00
3,600254,180659,England,26,France,25,4,3,3,2,...,0.94,4,4,1.00,5,5,1.00,9,9,1.00
4,600250,180659,France,43,Wales,0,7,0,4,0,...,0.97,2,3,0.66,9,10,0.90,7,8,0.87


In [1]:
player_dfs = []
for game in tqdm(range(len(all_teams_df.iloc[:5]))):
    try:
        player_dfs.append(
            match_scrape.get_player_stats(
                game_id=all_teams_df.iloc[game]['game_id'],
                league_id=all_teams_df.iloc[game]['league_id'])
        )
    except:
        print("Skipping game {}".format(all_teams_df.iloc[game]['game_id']))

NameError: name 'tqdm' is not defined

In [6]:
match_scrape.start_up_driver()

AttributeError: 'scrape_model' object has no attribute 'heads'

In [ ]:
match_scrape == None

False

In [6]:
all_teams_df.iloc[0]['game_id']

'597390'

In [7]:
match_scrape.get_player_stats(
    game_id=all_teams_df.iloc[0]['game_id'],
    league_id=all_teams_df.iloc[0]['league_id']
    )

[<selenium.webdriver.remote.webelement.WebElement (session="6451bce58ce6b05c138871674de9ff9f", element="f.62D6327B70A108E08643DF242C7BBE2E.d.E00A4102B133883BBF368AEF3510E30E.e.36")>, <selenium.webdriver.remote.webelement.WebElement (session="6451bce58ce6b05c138871674de9ff9f", element="f.62D6327B70A108E08643DF242C7BBE2E.d.E00A4102B133883BBF368AEF3510E30E.e.37")>]
0


IndexError: list index out of range

In [9]:
match_scrape.close_out_driver()

MaxRetryError: HTTPConnectionPool(host='localhost', port=56863): Max retries exceeded with url: /session/6451bce58ce6b05c138871674de9ff9f/window (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x76991bcd9940>: Failed to establish a new connection: [Errno 111] Connection refused'))